In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Optional, Dict
from abc import ABC, abstractmethod
import seaborn as sns


class BanditAgent(ABC):
    def __init__(self, n_features: int, alpha: float = 0.01):
        self.n_features = n_features
        self.alpha = alpha
        self.weights = np.zeros(n_features)
        self.name = "Base Agent"

    def reset(self):
        self.weights = np.zeros(self.n_features)

    def predict(self, x: np.ndarray) -> float:
        return np.dot(self.weights, x)

    def update_weights(self, x: np.ndarray, mask: np.ndarray,
                      predicted: float, actual: float,
                      arm_idx: Optional[int] = None):
        delta = actual - predicted
        update = self.alpha * delta * (mask * x)
        self.weights += update

    def post_update(self, arm_idx: int, actual: float):
        pass

    @abstractmethod
    def select_arm(self, candidates: List[Tuple[np.ndarray, np.ndarray]],
                   t: int) -> int:
        pass


class EpsilonGreedyAgent(BanditAgent):
    def __init__(self, n_features: int, alpha: float = 0.01, epsilon: float = 0.1):
        super().__init__(n_features, alpha)
        self.epsilon = epsilon
        self.name = f"ε-жадный (ε={epsilon})"

    def reset(self):
        super().reset()

    def select_arm(self, candidates: List[Tuple[np.ndarray, np.ndarray]],
                   t: int) -> int:
        n_arms = len(candidates)
        if np.random.random() < self.epsilon:
            return np.random.randint(n_arms)
        predictions = [self.predict(x) for x, _ in candidates]
        return int(np.argmax(predictions))


class LinUCBAgent(BanditAgent):
    def __init__(self, n_features: int, alpha: float = 0.01,
                 beta: float = 2.0, lambda_reg: float = 1.0):
        super().__init__(n_features, alpha)
        self.beta = beta
        self.lambda_reg = lambda_reg
        self.name = f"LinUCB (β={beta}, λ={lambda_reg})"

        self.A = np.eye(n_features) * lambda_reg
        self.b = np.zeros(n_features)

    def reset(self):
        super().reset()
        self.A = np.eye(self.n_features) * self.lambda_reg
        self.b = np.zeros(self.n_features)

    def get_weights(self) -> np.ndarray:
        return np.linalg.solve(self.A, self.b)

    def predict(self, x: np.ndarray) -> float:
        w = self.get_weights()
        return np.dot(w, x)

    def get_ucb_bonus(self, x: np.ndarray) -> float:
        A_inv = np.linalg.inv(self.A)
        return self.beta * np.sqrt(max(x @ A_inv @ x, 0))

    def get_ucb_value(self, x: np.ndarray) -> float:
        return self.predict(x) + self.get_ucb_bonus(x)

    def select_arm(self, candidates: List[Tuple[np.ndarray, np.ndarray]],
                   t: int) -> int:
        ucb_values = [self.get_ucb_value(x) for x, _ in candidates]
        return int(np.argmax(ucb_values))

    def update_weights(self, x: np.ndarray, mask: np.ndarray,
                      predicted: float, actual: float,
                      arm_idx: Optional[int] = None):
        self.A += np.outer(x, x)
        self.b += actual * x

    def post_update(self, arm_idx: int, actual: float):
        pass


class ThompsonSamplingAgent(BanditAgent):
    def __init__(self, n_features: int, alpha: float = 0.01):
        super().__init__(n_features, alpha)
        self.means: List[float] = []
        self.m2: List[float] = []
        self.n_pulls: List[int] = []
        self.name = "Thompson Sampling"

    def reset(self):
        super().reset()
        self.means = []
        self.m2 = []
        self.n_pulls = []

    def select_arm(self, candidates: List[Tuple[np.ndarray, np.ndarray]],
                   t: int) -> int:
        n_arms = len(candidates)
        while len(self.means) < n_arms:
            self.means.append(0.0)
            self.m2.append(0.0)
            self.n_pulls.append(0)

        samples = []
        for i in range(n_arms):
            n = self.n_pulls[i]
            variance = self.m2[i] / max(n - 1, 1) if n >= 2 else 1.0
            std = np.sqrt(max(variance, 1e-10))
            samples.append(np.random.normal(self.means[i], std))

        return int(np.argmax(samples))

    def post_update(self, arm_idx: int, actual: float):
        if arm_idx >= len(self.n_pulls):
            return
        n = self.n_pulls[arm_idx] + 1
        old_mean = self.means[arm_idx]
        self.means[arm_idx] = old_mean + (actual - old_mean) / n
        if n > 1:
            self.m2[arm_idx] += (actual - old_mean) * (actual - self.means[arm_idx])
        self.n_pulls[arm_idx] = n


class GradientBanditAgent(BanditAgent):
    def __init__(self, n_features: int, alpha: float = 0.01,
                 temperature: float = 1.0, baseline: bool = True):
        super().__init__(n_features, alpha)
        self.temperature = temperature
        self.baseline = baseline
        self.preferences: List[float] = []
        self.avg_reward = 0.0
        self.step = 0
        self.name = f"Gradient (T={temperature})"

    def reset(self):
        super().reset()
        self.preferences = []
        self.avg_reward = 0.0
        self.step = 0

    def select_arm(self, candidates: List[Tuple[np.ndarray, np.ndarray]],
                   t: int) -> int:
        n_arms = len(candidates)
        while len(self.preferences) < n_arms:
            self.preferences.append(0.0)

        prefs = np.array(self.preferences, dtype=np.float64)
        prefs = prefs - np.max(prefs)
        exp_prefs = np.exp(self.temperature * prefs)
        probs = exp_prefs / (np.sum(exp_prefs) + 1e-10)
        return int(np.random.choice(n_arms, p=probs))

    def post_update(self, arm_idx: int, actual: float):
        self.step += 1
        if self.baseline:
            self.avg_reward += (actual - self.avg_reward) / self.step
            baseline = self.avg_reward
        else:
            baseline = 0.0

        n_arms = len(self.preferences)
        prefs = np.array(self.preferences, dtype=np.float64)
        prefs = prefs - np.max(prefs)
        exp_prefs = np.exp(self.temperature * prefs)
        probs = exp_prefs / (np.sum(exp_prefs) + 1e-10)

        for i in range(n_arms):
            if i == arm_idx:
                self.preferences[i] += self.alpha * (actual - baseline) * (1 - probs[i])
            else:
                self.preferences[i] -= self.alpha * (actual - baseline) * probs[i]



class SimulationEnvironment:
    def __init__(self, n_speakers: int, n_features: int,
                 feature_distribution: str = 'uniform',
                 random_seed: Optional[int] = None):
        if random_seed is not None:
            np.random.seed(random_seed)

        self.n_speakers = n_speakers
        self.n_features = n_features
        self.features = []
        self.masks = []

        for _ in range(n_speakers):
            if feature_distribution == 'uniform':
                x = np.random.uniform(-1, 1, n_features)
            else:
                x = np.random.randn(n_features)
            mask = (np.abs(x) > 0.1).astype(float)
            self.features.append(x)
            self.masks.append(mask)

        self.true_weights = np.random.randn(n_features)
        self.theoretical_max = max(np.dot(self.true_weights, x)
                                   for x in self.features)
        self.true_rewards = [np.dot(self.true_weights, x)
                            for x in self.features]

    def get_candidates(self):
        return [(self.features[i], self.masks[i])
                for i in range(self.n_speakers)]

    def get_reward(self, speaker_idx: int) -> float:
        true_reward = np.dot(self.true_weights, self.features[speaker_idx])
        noise = np.random.normal(0, 0.1)
        return true_reward + noise

    def get_optimal_reward(self) -> float:
        return max(self.true_rewards)



def run_convergence_experiment(agent: BanditAgent,
                               env: SimulationEnvironment,
                               n_steps: int = 500) -> Dict:
    agent.reset()

    metrics = {
        'weight_errors': [],
        'prediction_errors': [],
        'cumulative_regret': [],
        'instant_regret': [],
        'rewards': [],
        'selected_arms': []
    }

    cumulative_regret = 0
    optimal_reward = env.get_optimal_reward()

    for t in range(1, n_steps + 1):
        candidates = env.get_candidates()
        chosen_idx = agent.select_arm(candidates, t)

        reward = env.get_reward(chosen_idx)
        x_chosen, mask_chosen = candidates[chosen_idx]

        predicted = agent.predict(x_chosen)
        true_pred = np.dot(env.true_weights, x_chosen)


        agent.update_weights(x_chosen, mask_chosen, predicted, reward,
                           arm_idx=chosen_idx)
        agent.post_update(chosen_idx, reward)


        regret = optimal_reward - reward
        cumulative_regret += regret

        current_preds = [agent.predict(x) for x, _ in candidates]
        true_preds = [np.dot(env.true_weights, x) for x, _ in candidates]

        metrics['weight_errors'].append(
            np.linalg.norm(agent.weights - env.true_weights))
        metrics['prediction_errors'].append(
            np.mean(np.abs(np.array(current_preds) - np.array(true_preds))))
        metrics['cumulative_regret'].append(cumulative_regret)
        metrics['instant_regret'].append(regret)
        metrics['rewards'].append(reward)
        metrics['selected_arms'].append(chosen_idx)


    for key in metrics:
        metrics[key] = np.array(metrics[key])

    return metrics


def run_multiple_experiments(agent_class, agent_params: Dict,
                             env_params: Dict,
                             n_experiments: int = 20,
                             n_steps: int = 500,
                             random_seed: int = 42) -> Dict:

    all_metrics = []

    for exp in range(n_experiments):
        exp_seed = random_seed + exp
        env = SimulationEnvironment(**env_params, random_seed=exp_seed)
        agent = agent_class(**agent_params)

        metrics = run_convergence_experiment(agent, env, n_steps)
        all_metrics.append(metrics)

    # Усреднение
    def average_array(key):
        return np.mean([m[key] for m in all_metrics], axis=0)

    def std_array(key):
        return np.std([m[key] for m in all_metrics], axis=0)

    return {
        'weight_errors_mean': average_array('weight_errors'),
        'weight_errors_std': std_array('weight_errors'),
        'prediction_errors_mean': average_array('prediction_errors'),
        'prediction_errors_std': std_array('prediction_errors'),
        'cumulative_regret_mean': average_array('cumulative_regret'),
        'cumulative_regret_std': std_array('cumulative_regret'),
        'rewards_mean': average_array('rewards'),
        'rewards_std': std_array('rewards'),
        'final_weight_error': average_array('weight_errors')[-1],
        'final_pred_error': average_array('prediction_errors')[-1],
        'total_regret': average_array('cumulative_regret')[-1],
        'avg_reward': average_array('rewards')[-1],
        'agent_name': agent_class(**agent_params).name,
        'n_experiments': n_experiments
    }




def plot_convergence_comparison(results: Dict[str, Dict],
                               n_steps: int,
                               save_path: Optional[str] = None):
    """Построение графиков сравнения сходимости"""
    sns.set_style("whitegrid")
    plt.rcParams['font.size'] = 10

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    steps = np.arange(1, n_steps + 1)


    colors = plt.cm.tab10(np.linspace(0, 1, len(results)))


    ax = axes[0]
    for (name, res), color in zip(results.items(), colors):
        ax.plot(steps, res['weight_errors_mean'],
               label=name, color=color, linewidth=2)
    ax.set_xlabel('Шаг')
    ax.set_ylabel('||w_t - w*||')
    ax.set_yscale('log')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

    ax = axes[1]
    for (name, res), color in zip(results.items(), colors):
        ax.plot(steps, res['prediction_errors_mean'],
               label=name, color=color, linewidth=2)
    ax.set_xlabel('Шаг')
    ax.set_ylabel('Среднее |ŷ_t - y*|')
    ax.set_yscale('log')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)



    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f" График сохранён: {save_path}")

    plt.show()


def print_summary_table(results: Dict[str, Dict]):

    print("\n" + "="*80)
    print("СВОДНАЯ ТАБЛИЦА СХОДИМОСТИ")
    print("="*80)
    print(f"{'Алгоритм':<25} {'Ошибка весов':>14} {'Ошибка предсказ.':>18} "
          f"{'Регрет':>12} {'Награда':>10}")
    print("-"*80)

    for name, res in results.items():
        print(f"{name:<25} {res['final_weight_error']:>14.4f} "
              f"{res['final_pred_error']:>18.4f} "
              f"{res['total_regret']:>12.2f} "
              f"{res['avg_reward']:>10.4f}")

    print("="*80)



if __name__ == "__main__":


    N_FEATURES = 7
    N_SPEAKERS = 10
    N_STEPS = 500
    N_EXPERIMENTS = 250
    RANDOM_SEED = 42



    env_params = {
        'n_speakers': N_SPEAKERS,
        'n_features': N_FEATURES,
        'feature_distribution': 'uniform'
    }


    agent_configs = [
        (EpsilonGreedyAgent, {
            'n_features': N_FEATURES,
            'alpha': 0.01,
            'epsilon': 0.1
        }),
        (EpsilonGreedyAgent, {
            'n_features': N_FEATURES,
            'alpha': 0.01,
            'epsilon': 0.3
        }),
        (LinUCBAgent, {
            'n_features': N_FEATURES,
            'alpha': 0.01,
            'beta': 2.0,
            'lambda_reg': 1.0
        }),
        (ThompsonSamplingAgent, {
            'n_features': N_FEATURES,
            'alpha': 0.01
        }),
    ]


    results = {}
    for agent_class, agent_params in agent_configs:
        agent_name = agent_class(**agent_params).name

        exp_results = run_multiple_experiments(
            agent_class, agent_params, env_params,
            n_experiments=N_EXPERIMENTS,
            n_steps=N_STEPS,
            random_seed=RANDOM_SEED
        )

        temp_env = SimulationEnvironment(**env_params, random_seed=RANDOM_SEED)
        exp_results['optimal_reward'] = temp_env.get_optimal_reward()

        results[agent_name] = exp_results
        print(f"   Ошибка весов: {exp_results['final_weight_error']:.4f}")
        print(f"   Регрет: {exp_results['total_regret']:.2f}\n")


    print_summary_table(results)

    plot_convergence_comparison(results, N_STEPS,
                               save_path='convergence_comparison.png')


    test_agent = LinUCBAgent(n_features=N_FEATURES, beta=2.0, lambda_reg=1.0)
    for n_exp in [5, 10, 20, 50]:
        exp_results = run_multiple_experiments(
            LinUCBAgent,
            {'n_features': N_FEATURES, 'beta': 2.0, 'lambda_reg': 1.0},
            env_params,
            n_experiments=n_exp,
            n_steps=N_STEPS,
            random_seed=RANDOM_SEED
        )

        ci_width = 1.96 * exp_results['weight_errors_std'][-1] / np.sqrt(n_exp)
        rel_error = ci_width / exp_results['final_weight_error'] * 100

        print(f"Экспериментов: {n_exp:2d} | "
              f"Ошибка весов: {exp_results['final_weight_error']:.4f} | "
              f"95% ДИ: ±{ci_width:.4f} | "
              f"Отн. погрешность: {rel_error:.1f}%")